# Resident Outcome Models — Inference

Loads all three resident outcome models and writes predictions to `operational.resident_outcome_predictions`.

| Column | Description |
|---|---|
| `predicted_reintegration_outcome` | Completed / In Progress / Not Started / On Hold |
| `predicted_reintegration_type` | Foster Care / Family Reunification / etc. |
| `predicted_edu_completion` | Completed / InProgress / NotStarted |

In [ ]:
import sys
sys.path.insert(0, '../jobs')

import json
from datetime import datetime, timezone

import joblib
import pandas as pd

from config import (
    EDUCATION_METADATA_PATH, EDUCATION_MODEL_PATH,
    OPERATIONAL_SCHEMA, WAREHOUSE_SCHEMA,
    REINTEGRATION_OUTCOME_METADATA_PATH, REINTEGRATION_OUTCOME_MODEL_PATH,
    REINTEGRATION_TYPE_METADATA_PATH, REINTEGRATION_TYPE_MODEL_PATH,
)
from utils_db import ensure_resident_outcome_predictions_table, get_engine, pg_conn

print('Imports loaded.')

## Load models

In [ ]:
outcome_model = joblib.load(str(REINTEGRATION_OUTCOME_MODEL_PATH))
type_model    = joblib.load(str(REINTEGRATION_TYPE_MODEL_PATH))
edu_model     = joblib.load(str(EDUCATION_MODEL_PATH))

with open(REINTEGRATION_OUTCOME_METADATA_PATH) as f: outcome_meta = json.load(f)
with open(REINTEGRATION_TYPE_METADATA_PATH)    as f: type_meta    = json.load(f)
with open(EDUCATION_METADATA_PATH)             as f: edu_meta     = json.load(f)

print(f"Outcome model v{outcome_meta['model_version']} | classes: {outcome_meta['classes']}")
print(f"Type   model v{type_meta['model_version']}    | classes: {type_meta['classes']}")
print(f"Edu    model v{edu_meta['model_version']}     | classes: {edu_meta['classes']}")

## Load feature data from warehouse

In [ ]:
engine = get_engine(WAREHOUSE_SCHEMA)
df = pd.read_sql('SELECT * FROM fact_resident_outcomes_ml', engine)
print(f'Scoring {len(df)} residents')
df.head()

## Run predictions

In [ ]:
outcome_features = outcome_meta['feature_cols']
type_features    = type_meta['feature_cols']
edu_features     = edu_meta['feature_cols']
outcome_classes  = outcome_meta['classes']
edu_classes      = edu_meta['classes']

outcome_probs = outcome_model.predict_proba(df[outcome_features])
outcome_preds = outcome_model.predict(df[outcome_features])
type_preds    = type_model.predict(df[type_features])
edu_probs     = edu_model.predict_proba(df[edu_features])
edu_preds     = edu_model.predict(df[edu_features])

outcome_prob_df = pd.DataFrame(outcome_probs, columns=outcome_classes)
edu_prob_df     = pd.DataFrame(edu_probs,     columns=edu_classes)
ts = datetime.now(timezone.utc).isoformat()

rows = []
for i, (_, row) in enumerate(df.iterrows()):
    rows.append((
        int(row['resident_id']),
        str(outcome_preds[i]),
        float(outcome_prob_df.iloc[i].get('Completed',   0)),
        float(outcome_prob_df.iloc[i].get('In Progress', 0)),
        float(outcome_prob_df.iloc[i].get('Not Started', 0)),
        float(outcome_prob_df.iloc[i].get('On Hold',     0)),
        str(type_preds[i]),
        str(edu_preds[i]),
        float(edu_prob_df.iloc[i].get('Completed',   0)),
        float(edu_prob_df.iloc[i].get('InProgress',  0)),
        float(edu_prob_df.iloc[i].get('NotStarted',  0)),
        ts,
    ))

print(f'Built {len(rows)} prediction rows')

## Write to database

In [ ]:
ensure_resident_outcome_predictions_table(OPERATIONAL_SCHEMA)

with pg_conn(OPERATIONAL_SCHEMA) as conn:
    with conn.cursor() as cur:
        cur.executemany("""
            INSERT INTO operational.resident_outcome_predictions (
                resident_id, predicted_reintegration_outcome,
                prob_completed, prob_in_progress, prob_not_started, prob_on_hold,
                predicted_reintegration_type, predicted_edu_completion,
                prob_edu_completed, prob_edu_in_progress, prob_edu_not_started, prediction_ts
            ) VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
            ON CONFLICT (resident_id) DO UPDATE SET
                predicted_reintegration_outcome = EXCLUDED.predicted_reintegration_outcome,
                prob_completed                  = EXCLUDED.prob_completed,
                prob_in_progress                = EXCLUDED.prob_in_progress,
                prob_not_started                = EXCLUDED.prob_not_started,
                prob_on_hold                    = EXCLUDED.prob_on_hold,
                predicted_reintegration_type    = EXCLUDED.predicted_reintegration_type,
                predicted_edu_completion        = EXCLUDED.predicted_edu_completion,
                prob_edu_completed              = EXCLUDED.prob_edu_completed,
                prob_edu_in_progress            = EXCLUDED.prob_edu_in_progress,
                prob_edu_not_started            = EXCLUDED.prob_edu_not_started,
                prediction_ts                   = EXCLUDED.prediction_ts
        """, rows)

print(f'Wrote {len(rows)} predictions to resident_outcome_predictions')
print(f'Timestamp: {ts}')